In [19]:
import requests
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup 
from datetime import datetime
import sqlite3
# url="https://en.wikipedia.org/wiki/Formula_One#Circuits"
# headers = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
# }
# data=requests.get(url,headers=headers)
# with open("html/page2.html","w",encoding="utf-8") as f:
#     f.write(data.text)
# data.status_code
# if data.status_code==200:
with open ("html/page1.html",'r',encoding="utf-8") as f:
    content=f.read()
soup=BeautifulSoup(content,"html.parser")
all_table=soup.find_all("table",class_="wikitable")
print(f"Total found table is {len(all_table)} ")
if len(all_table)>2:
    html_str=str(all_table[2])
    # here we are using StringIO because the pd can't able to find the table 
    # beacause of pd find the file of table so stringio wrap table into file infornt of pandas
    df2=pd.read_html(StringIO(html_str))[0]
print(df2.head())
df_contract=df2.copy()
# we are copy df because escape from data harm
df_contract['Contract ends']=df_contract['Contract ends'].str.replace(r'\[.*\]',"",regex=True)
df_contract["Contract ends"]=pd.to_numeric(df_contract['Contract ends'] ,errors='coerce')
data=df_contract["Contract ends"]
# df_contract["country"] = df_contract["Grand Prix"].str.replace("Grand Prix","",case=False).str.strip()
df_contract["mix_data"] = df_contract["Grand Prix"].str.replace("Grand Prix","",case=False).str.strip()
df_contract["country"] = df_contract["Circuit"].str.replace("Circuit","",case=False).str.split(',').str[-1].str.strip()

country_clean_mapping = {
        "Abu Dhabi": "UAE",
        "Montmeló": "Spain",
        "Stavelot": "Belgium",
        "Silverstone": "UK",
        "Montreal": "Canada",
        "Mogyoród": "Hungary",
        "Monza": "Italy",
        "Paradise, Nevada": "USA",
        "Miami Gardens, Florida": "USA",
        "Austin, Texas": "USA",
        "Madrid": "Spain",
        "Melbourne": "Australia",
        "Spielberg": "Austria",
        "Baku": "Azerbaijan",
        "Shanghai": "China",
        "Zandvoort": "Netherlands",
        "Suzuka": "Japan",
        "Nevada": "USA",
        "Mexico City": "Mexico",
        "Florida": "USA",
        "Lusail": "Qatar",
        "São Paulo": "Brazil",
        "Singapore": "Singapore",
        "Texas": "USA"
    }
df_contract['country'] = df_contract['country'].replace(country_clean_mapping)

df_contract[['mix_data', 'country']].head(10)
df_contract=df_contract.drop("mix_data",axis=1)
total_circuits=len(df_contract)
current_date=datetime.now().year
risk=current_date+4
high_risk=df_contract[data<=risk]
pro_risk=len(high_risk)/total_circuits
print(f"The current year is  :{current_date}")
print(f"The high risk  is :{high_risk} \nIts probability is :{pro_risk:.2f}")


sql_conv=sqlite3.connect("f1_project.db")
sql_file=df_contract.to_sql("risk_analysis",sql_conv,if_exists="replace",index=False)

query="SELECT*FROM risk_analysis LIMIT 5"
Data=pd.read_sql_query(query,sql_conv)
print(Data)
sql_conv.close()

Total found table is 4 
                       Grand Prix                                   Circuit  \
0            Abu Dhabi Grand Prix             Yas Marina Circuit, Abu Dhabi   
1           Australian Grand Prix            Albert Park Circuit, Melbourne   
2             Austrian Grand Prix                  Red Bull Ring, Spielberg   
3           Azerbaijan Grand Prix                   Baku City Circuit, Baku   
4  Barcelona-Catalunya Grand Prix  Circuit de Barcelona-Catalunya, Montmeló   

  Contract ends   Ref.  
0          2030  [212]  
1          2037  [213]  
2          2041  [214]  
3          2030  [215]  
4       2032[f]  [216]  
The current year is  :2026
The high risk  is :                Grand Prix                                     Circuit  \
0     Abu Dhabi Grand Prix               Yas Marina Circuit, Abu Dhabi   
3    Azerbaijan Grand Prix                     Baku City Circuit, Baku   
8       Chinese Grand Prix    Shanghai International Circuit, Shanghai   
9        

In [20]:

df_contract

,Grand Prix,Circuit,Contract ends,Ref.,country
0,Abu Dhabi Grand Prix,"Yas Marina Circuit, Abu Dhabi",2030,[212],UAE
1,Australian Grand Prix,"Albert Park Circuit, Melbourne",2037,[213],Australia
2,Austrian Grand Prix,"Red Bull Ring, Spielberg",2041,[214],Austria
3,Azerbaijan Grand Prix,"Baku City Circuit, Baku",2030,[215],Azerbaijan
4,Barcelona-Catalunya Grand Prix,"Circuit de Barcelona-Catalunya, Montmeló",2032,[216],Spain
5,Belgian Grand Prix,"Circuit de Spa-Francorchamps, Stavelot",2031,[217],Belgium
6,British Grand Prix,"Silverstone Circuit, Silverstone",2034,[218],UK
7,Canadian Grand Prix,"Circuit Gilles Villeneuve, Montreal",2035,[219],Canada
8,Chinese Grand Prix,"Shanghai International Circuit, Shanghai",2030,[220],China
9,Dutch Grand Prix,"Circuit Zandvoort, Zandvoort",2026,[221],Netherlands
